#Instalación de librerias

In [ ]:
!pip install requests
!pip install tsplib95

In [ ]:
import random
import copy
import tsplib95


#Carga de datos del problema

In [ ]:
# Carga del problema
file = "swiss42.tsp"
problem = tsplib95.load(file)
print("Fichero cargado:", file)

# Nodos
Nodos = list(problem.get_nodes())

# Aristas
Aristas = list(problem.get_edges())
print(f"Nodos: {len(Nodos)}, Aristas: {len(Aristas)}")
print(f"Distancia(1,2) = {problem.get_weight(1, 2)}")


#Funciones de la Actividad Guiada 3

In [ ]:
#Funciones de la Actividad Guiada 3
#Se genera una solucion aleatoria con comienzo en en el nodo 0
def crear_solucion(Nodos):
  solucion = [Nodos[0]]
  for n in Nodos[1:] :
    solucion = solucion + [random.choice(list(set(Nodos) - set(solucion)))]
  return solucion

#Devuelve la distancia entre dos nodos
def distancia(a,b, problem):
  return problem.get_weight(a,b)


#Devuelve la distancia total de una trayectoria/solucion
def distancia_total(solucion, problem):
  distancia_total = 0
  for i in range(len(solucion)-1):
    distancia_total += distancia(solucion[i] ,solucion[i+1] ,  problem)
  return distancia_total + distancia(solucion[len(solucion)-1] ,solucion[0], problem)

#Funciones Auxiliares

In [ ]:
#Genera una poblacion inicial de soluciones de tamaño N.
# Puede ser válida la solución aleatoria de la AG3: crear_solucion(Nodos)
def generar_poblacion(Nodos, N):
  poblacion = []
  for _ in range(N):
    poblacion.append(crear_solucion(Nodos))
  return poblacion

In [ ]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
  mejor_solucion = None
  mejor_distancia = float('inf')

  for individuo in poblacion:
    dist = distancia_total(individuo, problem)
    if dist < mejor_distancia:
      mejor_distancia = dist
      mejor_solucion = individuo

  return (mejor_solucion, mejor_distancia)

In [ ]:
#Funcion de cruce. Recibe una poblacion(lista de soluciones) y devuelve la población ampliada con los hijos.
# Todos los individuos de la población son selecionados para el cruce(si la población es par)
# Podría aplicarse un proceso previo de selección para elegir los individuos que se desea cruzar.
def Cruzar(poblacion, mutacion, problem):
  hijos = []
  # Mezclar la poblacion para emparejar al azar
  indices = list(range(len(poblacion)))
  random.shuffle(indices)

  # Emparejar de dos en dos
  for i in range(0, len(indices) - 1, 2):
    padre1 = copy.deepcopy(poblacion[indices[i]])
    padre2 = copy.deepcopy(poblacion[indices[i+1]])
    descendientes = Descendencia([padre1, padre2], problem, mutacion)
    hijos.extend(descendientes)

  # Devolver poblacion ampliada con los hijos
  return poblacion + hijos

In [ ]:
#Funcion para generar hijos a partir de 2 padres:
# Se elige el metodo de 1-punto de corte pero es posible usar otros n-puntos, uniforme, dependiendo del problema
def Descendencia(padres, problem, mutacion):
  padre1 = padres[0]
  padre2 = padres[1]

  # Punto de corte aleatorio (evitando posicion 0 que siempre es el nodo inicio)
  punto = random.randint(1, len(padre1) - 1)

  # Cruce de 1 punto: primera parte de un padre + segunda parte del otro
  hijo1 = padre1[:punto] + padre2[punto:]
  hijo2 = padre2[:punto] + padre1[punto:]

  # Los hijos no son factibles (nodos repetidos/faltantes), hay que repararlos
  hijo1 = Factibilizar(hijo1, problem)
  hijo2 = Factibilizar(hijo2, problem)

  # Aplicar mutacion
  hijo1 = Mutar(hijo1, mutacion)
  hijo2 = Mutar(hijo2, mutacion)

  return [hijo1, hijo2]

In [ ]:
#Para el operador de cruce 1-punto los hijos generados no son soluciones(algunos nodos se repiten y otros no están)
def Factibilizar(solucion, problem):
  Nodos = list(problem.get_nodes())

  # Detectar nodos repetidos y faltantes
  vistos = set()
  repetidos_idx = []  # Indices donde hay nodos repetidos
  for i in range(len(solucion)):
    if solucion[i] in vistos:
      repetidos_idx.append(i)
    else:
      vistos.add(solucion[i])

  # Nodos que faltan en la solucion
  faltantes = list(set(Nodos) - vistos)
  random.shuffle(faltantes)

  # Sustituir nodos repetidos por los faltantes
  for i, idx in enumerate(repetidos_idx):
    solucion[idx] = faltantes[i]

  return solucion

In [ ]:
#Funcion de mutación. Se eligen dos nodos y se intercambia. Se podrian añadir otros operaradores
# Se hace mutaciones mutacion% de las veces
def Mutar(solucion, mutacion):
  if random.random() < mutacion:
    # Elegir dos posiciones aleatorias (evitando la 0 que es el nodo inicio)
    i = random.randint(1, len(solucion) - 1)
    j = random.randint(1, len(solucion) - 1)
    # Intercambiar los dos nodos
    solucion[i], solucion[j] = solucion[j], solucion[i]
  return solucion

In [ ]:
#Funcion de seleccion de la población. Recibe como parametro una poblacion y
# devuelve una poblacion a la que se ha eliminado individuos poco aptos(fitness alto) y para mantener una poblacion estable de N individuos
#Se tiene en cuenta el porcentaje elitismo pasado como parametro
# Para los individuos que no son de la elite podríamos usar una selección de ruleta(proporcional a su fitness)
def Seleccionar(problem, poblacion, N, elitismo):
  # Ordenar poblacion por fitness (distancia menor = mejor)
  poblacion_evaluada = [(ind, distancia_total(ind, problem)) for ind in poblacion]
  poblacion_evaluada.sort(key=lambda x: x[1])

  # Cantidad de individuos elite que se mantienen directamente
  n_elite = max(1, int(N * elitismo))

  # Los elite pasan directamente
  nueva_poblacion = [ind for ind, dist in poblacion_evaluada[:n_elite]]

  # El resto se selecciona por ruleta (proporcional al inverso de la distancia)
  restantes = poblacion_evaluada[n_elite:]
  n_restantes = N - n_elite

  if len(restantes) > 0 and n_restantes > 0:
    # Fitness inversamente proporcional a la distancia
    fitness = [1.0 / dist for ind, dist in restantes]
    suma_fitness = sum(fitness)
    probabilidades = [f / suma_fitness for f in fitness]

    for _ in range(n_restantes):
      r = random.random()
      acumulado = 0
      for idx, prob in enumerate(probabilidades):
        acumulado += prob
        if r <= acumulado:
          nueva_poblacion.append(restantes[idx][0])
          break

  return nueva_poblacion

#Proceso Principal

In [ ]:
#Funcion principal del algoritmo genetico
#######################################################3
def algoritmo_genetico(problem=problem,N=100,mutacion=.15,elitismo=.1,generaciones=100):
  # problem = datos del problema
  # N = Tamaño de la población
  # mutacion = probabilidad de una mutación
  # elitismo = porcion de la mejor poblacion a mantener
  # generaciones = nº de generaciones a generar para finalizar

  #Genera la poblacion inicial
  Nodos = list(problem.get_nodes())
  poblacion = generar_poblacion(Nodos,N)

  #Inicializamos valores para la mejor solucion
  (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)


  #Condicion de parada
  parar = False
  n=0
  #Inciamos el cliclo de generaciones
  while(parar == False) :

    #Cruce de la poblacion(incluye mutación)
    poblacion = Cruzar(poblacion, mutacion, problem)

    #Seleccionamos la población
    poblacion = Seleccionar(problem,poblacion, N, elitismo)

    #Evaluamos la nueva población
    (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)

    print("Generacion #", n, "\nLa mejor solución es:" , mejor_solucion, "\ncon distancia " , mejor_distancia, "\n")

    #Numero de generaciones. Criterio de parada
    if n==generaciones:
      parar = True
    n +=1

  return mejor_solucion


sol = algoritmo_genetico(problem=problem,N=500,mutacion=.3,elitismo=.40,generaciones=250)